# AAI 510 — Assignment 3

## Building Agent Tools

**In this lab you will:**
- **Required (Sections 1–5):** Create **Unity Catalog function tools** — one SQL function and one Python function — and test them.
- **Required (Section 6):** Set up **semantic search with embeddings** on the UltraFeedback dataset so an agent can find similar instructions by meaning.
- **Required (Section 7):** Configure an **external MCP server** (You.com web search) in Databricks so your agent can access live web information.
- **Optional, strongly encouraged (Section 8):** Create an **Agent Skill** (`SKILL.md`) that documents the tools you built.

### The big picture

Over Weeks 3–5 you are building an **UltraFeedback Expert** agent — an AI assistant that helps users explore and understand LLM preference data. This week you create the **tools**; next week you wire them into a working agent; in Week 5 you evaluate how well it performs.

| Week | What you do | Deliverable |
|------|------------|-------------|
| 3 (this week) | Build tools: UC functions, semantic search, MCP | Tested tools + MCP config |
| 4 | Wire tools into an agent; register a prompt; compare LLMs | Working agent |
| 5 | Evaluate the agent with judges and an eval dataset | Evaluation report |

**Readings this week:**
- [Practical Guide for Agentic AI Workflows](https://arxiv.org/pdf/2512.08769)
- [MCP Architecture](https://modelcontextprotocol.io/docs/learn/architecture)

**Key docs:**
- [Create AI agent tools with UC functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool)
- [Foundation Model APIs — Embeddings](https://docs.databricks.com/en/machine-learning/model-serving/score-foundation-models.html)
- [External MCP Servers on Databricks](https://docs.databricks.com/aws/en/generative-ai/mcp/external-mcp)
- [You.com on Databricks Marketplace](https://marketplace.databricks.com/details/32d5b7b6-0fab-4bba-9c57-5de23dd58996/Youcom_Youcom-MCP-The-1-AI-Web-Search-API)

---
## 1. Why agents need tools *(Required)*

An LLM on its own can only generate text. **Tools** give agents the ability to *act* — query databases, search the web, look up facts, run computations. In this assignment you'll create three kinds of tools:

| Tool type | What it does | Example |
|-----------|-------------|--------|
| **UC SQL function** | Deterministic lookup against structured data | "How many rows come from `evol_instruct`?" |
| **UC Python function** | Custom computation or text processing | "Analyze the complexity of this instruction" |
| **Semantic search** | Embedding-based similarity search over text | "Find instructions similar to *Explain quantum tunneling*" |
| **External MCP** | Access external services (web search, APIs) | "Search the web for recent LLM benchmarks" |

Each tool is registered in a place the agent can discover it — Unity Catalog for functions and semantic search, MCP for external services.

---
## 2. Install dependencies *(Required)*

We need two packages:
- `unitycatalog-ai[databricks]` — the Unity Catalog AI client for creating and testing UC functions as agent tools.
- `numpy` — for computing cosine similarity between embedding vectors.

**Docs:** [Unity Catalog AI](https://docs.unitycatalog.io/ai/) · [Foundation Model APIs](https://docs.databricks.com/en/machine-learning/model-serving/score-foundation-models.html)

In [0]:
# Install the UC AI client (for creating/testing UC functions as tools)
# and numpy (for computing cosine similarity in semantic search)
%pip install unitycatalog-ai[databricks] numpy
dbutils.library.restartPython()

In [0]:
import sys
import subprocess

# Force installation directly via system shell, bypassing Spark Connect completely
subprocess.check_call([sys.executable, "-m", "pip", "install", "unitycatalog-ai[databricks]", "numpy"])

# Let Databricks know it needs to refresh the internal sys.path pointers
dbutils.library.restartPython()

---
## 3. Verify your data *(Required)*

Confirm the UltraFeedback table from Assignment 1 is still available. If you get an error, re-run Assignment 1 first.

In [0]:
# Quick check: confirm the table exists, show schema and row count
df = spark.table("main.default.assignment_file")
print(f"Row count: {df.count():,}")
print(f"Columns:  {df.columns}")
df.printSchema()
display(df.limit(3))

In [0]:
import os
from databricks.connect import DatabricksSession

# 1. Clear out any cached connection environment variables in this notebook session
if "SPARK_REMOTE" in os.environ:
    del os.environ["SPARK_REMOTE"]

# 2. Force Databricks to completely terminate the stale session and spin up a new channel
spark = DatabricksSession.builder.getOrCreate()

# 3. Test if the connection is alive by reading just the metadata
print(f"Spark Version Connected: {spark.version}")

---
## 4. Create Unity Catalog function tools *(Required)*

Unity Catalog functions are UDFs registered in `catalog.schema.function_name`. When an agent needs a tool, it calls the function by name. Two patterns are common:

1. **SQL functions** — best for deterministic lookups against tables (e.g., counts, filters, joins).
2. **Python functions** — best for custom logic, text processing, or computations that don't map cleanly to SQL.

You'll create two SQL functions, a Python function, then build your own.

These functions exist in Unity Catalog, so you can see them in the UI after registration

**Docs:** [Create AI agent tools with UC functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool) · [CREATE FUNCTION syntax](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function)

### 4.1 UC SQL functions

#### all_model_combinations

This function shows all combinations of chosen & rejected models, sorted by the most common combinations.

**Key points:**
- The `COMMENT` on the function and its parameters helps the agent understand *when* and *how* to use the tool. Write clear, descriptive comments.
- The function returns a `TABLE` which has all the data in the query returned

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.all_model_combinations()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Shows all combinations of chosen and rejected models, counts and average ratings'
RETURN (
  SELECT 
    `chosen-model`,
    `rejected-model`,
    count(source) as records,
    avg(`chosen-rating`) as avg_chosen_rating,
    avg(`rejected-rating`) as avg_rejected_rating
  FROM main.default.assignment_file
  GROUP BY 1,2
  ORDER BY 3 desc
)

In [0]:
%sql
select * from main.default.all_model_combinations() limit 10

#### compare_models

The function below compares two specific models to see how frequently an individual model wins vs. loses.

This also returns a table with the count of each scenario, as well as the average winning and losing rating

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.compare_models(
  model_a STRING COMMENT 'First model name to compare e.g. gpt-4',
  model_b STRING COMMENT 'Second model name to compare'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Compares two models by how often each was chosen vs rejected in the UltraFeedback dataset. Returns a summary string with win counts for each model.'
RETURN (
  SELECT 
    CASE when `chosen-model` = model_a AND `rejected-model` = model_b THEN 'A_win'
         when `chosen-model` = model_b AND `rejected-model` = model_a THEN 'B_win'
         else 'other' end as comparison_scenario,
    count(*) as win_count,
    avg(`chosen-rating`) as avg_chosen_rating,
    avg(`rejected-rating`) as avg_rejected_rating
  FROM main.default.assignment_file
  WHERE `chosen-model` IN (model_a, model_b)
     AND `rejected-model` IN (model_a, model_b)
  GROUP BY 1
  ORDER BY 1
)

In [0]:
%sql
select * 
from main.default.compare_models('gpt-3.5-turbo', 'alpaca-7b')

In [0]:
%sql
select * 
from main.default.compare_models('vicuna-33b', 'wizardlm-13b')

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.compare_models(
  model_a STRING COMMENT 'First model name to compare e.g. gpt-4',
  model_b STRING COMMENT 'Second model name to compare'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Compares two models by how often each was chosen vs rejected in the assignment file dataset. Returns a summary with explicit winning and losing model names.'
RETURN (
  SELECT 
    CASE 
      WHEN `chosen-model` = model_a AND `rejected-model` = model_b THEN 'A_win'
      WHEN `chosen-model` = model_b AND `rejected-model` = model_a THEN 'B_win'
      ELSE 'other' 
    END AS comparison_scenario,
    
    -- Explicitly listing out the names of the winning and losing models for clarity
    `chosen-model` AS winning_model,
    `rejected-model` AS losing_model,
    
    COUNT(*) AS win_count,
    ROUND(AVG(`chosen-rating`), 2) AS avg_winning_rating,
    ROUND(AVG(`rejected-rating`), 2) AS avg_losing_rating
  FROM main.default.assignment_file
  WHERE `chosen-model` IN (model_a, model_b)
    AND `rejected-model` IN (model_a, model_b)
  GROUP BY 1, 2, 3
  ORDER BY 1
)

In [0]:
%sql
select * 
from main.default.compare_models('vicuna-33b', 'wizardlm-13b')

### 4.2 Python function: `analyze_instruction`

This function takes an instruction text and returns complexity metrics (word count, sentence count, estimated complexity level). An agent could use this to assess how complex a prompt is before deciding how to handle it.

**Key points:**
- Python UC functions must have **type hints** on all arguments and the return value.
- **Imports go inside the function body** — they won't be resolved otherwise.
- Use [Google-style docstrings](https://google.github.io/styleguide/pyguide.html#383-functions-and-methods) so the agent can parse the description.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# Initialize the Databricks Function Client
uc_client = DatabricksFunctionClient()

# Define the Python function with type hints and a clear docstring.
# NOTE: all imports must be INSIDE the function body.
def analyze_instruction(instruction: str) -> str:
    """
    Analyzes the complexity and characteristics of an instruction prompt.

    Returns word count, sentence count, average word length, estimated
    complexity level (low/medium/high), and whether the text is a question.
    Use this to assess instruction difficulty before generating a response.

    Args:
        instruction: The instruction or prompt text to analyze.

    Returns:
        A JSON string with analysis metrics.
    """
    import json
    import re

    words = instruction.split()
    word_count = len(words)
    sentences = [s.strip() for s in re.split(r'[.!?]+', instruction) if s.strip()]
    sentence_count = len(sentences)
    avg_word_length = round(sum(len(w) for w in words) / max(word_count, 1), 1)
    is_question = instruction.strip().endswith('?')

    if word_count > 50 or sentence_count > 3:
        complexity = "high"
    elif word_count > 20:
        complexity = "medium"
    else:
        complexity = "low"

    return json.dumps({
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_word_length": avg_word_length,
        "complexity": complexity,
        "is_question": is_question
    })

# Register the function in Unity Catalog (main.default schema)
function_info = uc_client.create_python_function(
    func=analyze_instruction,
    catalog="main",
    schema="default",
    replace=True  # overwrite if it already exists
)
print(f"Registered: {function_info.full_name}")

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

uc_client = DatabricksFunctionClient()

# Test the function out that we just created

result = uc_client.execute_function(
    function_name="main.default.analyze_instruction",
    parameters={"instruction": "Heisenberg explained resonance theory, which is a structure-activity relationship that describes the effect of complementary chemical groups on a drug's properties and actions (such as solubility, absorption, and therapeutic effects). These complementary groups can be used to reduce toxicity and increase the potency of drug compounds"})

print(result.value)

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

uc_client = DatabricksFunctionClient()

# Test the function out that we just created

result = uc_client.execute_function(
    function_name="main.default.analyze_instruction",
    parameters={"instruction": "Heisenberg explained resonance theory, which is a structure-activity relationship that describes the effect of complementary chemical groups on a drug's properties and actions (such as solubility, absorption, and therapeutic effects). These complementary groups can be used to reduce toxicity and increase the potency of drug compounds"})

print(result.value)

In [0]:
%sql
-- Create a SQL function to find the best model-source pairs
-- based on selection frequency and average rating

CREATE OR REPLACE FUNCTION main.default.get_best_model_source_pairs(
  min_selections INT COMMENT 'Minimum number of times a model must be chosen from a source to be included',
  top_n INT COMMENT 'Number of top pairs to return per source'
)
RETURNS TABLE(
  source STRING COMMENT 'Data source name',
  chosen_model STRING COMMENT 'Model that was chosen',
  selection_count BIGINT COMMENT 'Number of times this model was chosen for this source',
  avg_rating DOUBLE COMMENT 'Average rating when this model was chosen for this source',
  source_total BIGINT COMMENT 'Total selections from this source',
  percentage DOUBLE COMMENT 'Percentage of source selections',
  composite_score DOUBLE COMMENT 'Combined score: (avg_rating * 20) + log10(selection_count + 1) * 10'
)
LANGUAGE SQL
COMMENT 'Returns the top-performing model-source pairs ranked by composite score of rating and frequency. Use this to understand which models work best for specific data sources in the UltraFeedback dataset.'
RETURN (
  WITH source_model_stats AS (
    SELECT 
      source,
      `chosen-model` AS chosen_model,
      COUNT(*) AS selection_count,
      AVG(`chosen-rating`) AS avg_rating,
      SUM(COUNT(*)) OVER (PARTITION BY source) AS source_total
    FROM main.default.assignment_file
    GROUP BY source, `chosen-model`
    HAVING COUNT(*) >= min_selections
  ),
  ranked_pairs AS (
    SELECT 
      source,
      chosen_model,
      selection_count,
      ROUND(avg_rating, 3) AS avg_rating,
      source_total,
      ROUND((selection_count * 100.0 / source_total), 2) AS percentage,
      -- Composite score: weighted combination of rating and frequency
      -- Rating contributes 0-100 points (rating * 20), frequency contributes 0-30 points (log scale)
      ROUND((avg_rating * 20) + (LOG10(selection_count + 1) * 10), 2) AS composite_score,
      ROW_NUMBER() OVER (PARTITION BY source ORDER BY 
        (avg_rating * 20) + (LOG10(selection_count + 1) * 10) DESC
      ) AS rank_within_source
    FROM source_model_stats
  )
  SELECT 
    source,
    chosen_model,
    selection_count,
    avg_rating,
    source_total,
    percentage,
    composite_score
  FROM ranked_pairs
  WHERE rank_within_source <= top_n
  ORDER BY source, composite_score DESC
)

In [0]:
%sql
-- Test the function: Get top 3 model-source pairs with at least 100 selections
SELECT 
  source,
  chosen_model,
  selection_count,
  avg_rating,
  percentage AS pct_of_source,
  composite_score
FROM main.default.get_best_model_source_pairs(
  min_selections => 100,  -- Only pairs with 100+ selections
  top_n => 3              -- Top 3 per source
)
ORDER BY source, composite_score DESC;

In [0]:
%sql
-- Test the function: Get top 3 model-source pairs with at least 100 selections
SELECT 
  source,
  chosen_model,
  selection_count,
  avg_rating,
  percentage AS pct_of_source,
  composite_score
FROM main.default.get_best_model_source_pairs(
  min_selections => 100,  -- Only pairs with 100+ selections
  top_n => 3              -- Top 3 per source
)
ORDER BY source, composite_score DESC;

In [0]:
%sql
-- Drop the old Python function and create a SQL version instead
DROP FUNCTION IF EXISTS main.default.recommend_reasoning_agent_route;

-- Create a SQL function to recommend model-source routes for reasoning agents
-- Analyzes the UltraFeedback dataset to find models that excel at reasoning tasks

CREATE OR REPLACE FUNCTION main.default.recommend_reasoning_agent_route(
  use_case STRING COMMENT 'Type of reasoning task: medical_diagnosis, general_reasoning, or complex_analysis',
  min_rating DOUBLE COMMENT 'Minimum average rating threshold (1.0-5.0)',
  top_n INT COMMENT 'Number of top recommendations to return'
)
RETURNS TABLE(
  rank INT COMMENT 'Ranking position (1 = best)',
  model STRING COMMENT 'Recommended model name',
  source STRING COMMENT 'Data source the model excels on',
  avg_rating DOUBLE COMMENT 'Average quality rating',
  selection_count BIGINT COMMENT 'Number of times chosen',
  reasoning_score DOUBLE COMMENT 'Computed score for reasoning capability',
  recommendation STRING COMMENT 'Explanation of why this route is recommended'
)
LANGUAGE SQL
COMMENT 'Recommends optimal model-source combinations for reasoning agents that analyze patient feelings, review historical data, identify problems, and provide rated solutions. Focuses on complex reasoning tasks requiring multi-step analysis.'
RETURN (
  WITH source_preferences AS (
    -- Define which sources are relevant for each use case
    SELECT 'evol_instruct' AS source WHERE use_case IN ('medical_diagnosis', 'general_reasoning', 'complex_analysis')
    UNION ALL SELECT 'flan_v2_cot' WHERE use_case IN ('medical_diagnosis', 'general_reasoning')
    UNION ALL SELECT 'ultrachat' WHERE use_case IN ('medical_diagnosis', 'complex_analysis')
    UNION ALL SELECT 'flan_v2_niv2' WHERE use_case = 'general_reasoning'
    UNION ALL SELECT 'sharegpt' WHERE use_case = 'complex_analysis'
  ),
  model_analysis AS (
    SELECT 
      af.source,
      af.`chosen-model` AS model,
      COUNT(*) AS selection_count,
      AVG(af.`chosen-rating`) AS avg_rating
    FROM main.default.assignment_file af
    INNER JOIN source_preferences sp ON af.source = sp.source
    GROUP BY af.source, af.`chosen-model`
    HAVING AVG(af.`chosen-rating`) >= min_rating
  ),
  scored_recommendations AS (
    SELECT 
      model,
      source,
      selection_count,
      ROUND(avg_rating, 3) AS avg_rating,
      -- Compute reasoning score:
      -- Base: avg_rating * 20 (0-100 points)
      -- Bonus: +10 for evol_instruct (known for complex reasoning)
      -- Bonus: +5 for flan_v2_cot (chain-of-thought reasoning)
      -- Frequency factor: log10(selection_count + 1) * 5
      ROUND(
        (avg_rating * 20) +
        CASE WHEN source = 'evol_instruct' THEN 10 ELSE 0 END +
        CASE WHEN source = 'flan_v2_cot' THEN 5 ELSE 0 END +
        (LOG10(selection_count + 1) * 5),
        2
      ) AS reasoning_score,
      -- Generate recommendation text based on source
      CASE 
        WHEN source = 'evol_instruct' THEN 'Excellent for multi-step reasoning and complex problem-solving'
        WHEN source = 'flan_v2_cot' THEN 'Strong chain-of-thought reasoning, good for diagnostic workflows'
        WHEN source = 'ultrachat' THEN 'Conversational and empathetic, suitable for patient interaction'
        WHEN source = 'sharegpt' THEN 'Diverse conversational skills, adaptable to various scenarios'
        ELSE 'Reliable performance on reasoning tasks'
      END AS recommendation
    FROM model_analysis
  ),
  ranked_recommendations AS (
    SELECT 
      ROW_NUMBER() OVER (ORDER BY reasoning_score DESC, avg_rating DESC) AS rank,
      model,
      source,
      avg_rating,
      selection_count,
      reasoning_score,
      recommendation
    FROM scored_recommendations
  )
  SELECT 
    rank,
    model,
    source,
    avg_rating,
    selection_count,
    reasoning_score,
    recommendation
  FROM ranked_recommendations
  WHERE rank <= top_n
  ORDER BY rank
)

In [0]:
%sql
-- Test the reasoning agent recommendation function

-- Test 1: Medical diagnosis use case (top 5)
SELECT 
  'Medical Diagnosis Use Case' AS test_name,
  rank,
  model,
  source,
  avg_rating,
  selection_count,
  reasoning_score,
  recommendation
FROM main.default.recommend_reasoning_agent_route(
  use_case => 'medical_diagnosis',
  min_rating => 4.5,
  top_n => 5
)
ORDER BY rank;

In [0]:
%sql
-- Create a SQL function to find the best model-source pairs for a technical documentation agent
-- Use case: Agent reviews weld inspection reports, considers AWS D1.1 codes (RAG), writes final report

CREATE OR REPLACE FUNCTION main.default.find_best_technical_doc_models(
  min_rating DOUBLE COMMENT 'Minimum quality rating (e.g., 4.5)',
  top_n INT COMMENT 'Number of recommendations (e.g., 3)'
)
RETURNS TABLE(
  rank INT COMMENT 'Ranking (1 = best)',
  model STRING COMMENT 'Recommended model name',
  source STRING COMMENT 'Data source',
  avg_rating DOUBLE COMMENT 'Average quality rating',
  selection_count BIGINT COMMENT 'Times chosen',
  technical_score DOUBLE COMMENT 'Score for technical documentation capability',
  use_case_fit STRING COMMENT 'Why this model fits the weld inspection use case'
)
LANGUAGE SQL
COMMENT 'Finds top 3 models for technical documentation agent that reviews weld inspection reports, references codes (AWS D1.1), and writes final reports. Focuses on models good at document synthesis, technical writing, and multi-source analysis.'
RETURN (
  WITH technical_sources AS (
    -- Sources good for technical documentation: complex instructions, diverse content, explanatory
    SELECT 'evol_instruct' AS source  -- Multi-step reasoning, complex tasks
    UNION ALL SELECT 'sharegpt'      -- Diverse conversational, adaptable
    UNION ALL SELECT 'ultrachat'     -- Explanatory, detailed responses
    UNION ALL SELECT 'flan_v2_niv2'  -- Instructional, structured tasks
  ),
  model_analysis AS (
    SELECT 
      af.source,
      af.`chosen-model` AS model,
      COUNT(*) AS selection_count,
      AVG(af.`chosen-rating`) AS avg_rating
    FROM main.default.assignment_file af
    INNER JOIN technical_sources ts ON af.source = ts.source
    GROUP BY af.source, af.`chosen-model`
    HAVING AVG(af.`chosen-rating`) >= min_rating
  ),
  scored_models AS (
    SELECT 
      model,
      source,
      selection_count,
      ROUND(avg_rating, 3) AS avg_rating,
      -- Technical documentation score:
      -- Base: avg_rating * 20 (0-100 points)
      -- Bonus: +15 for evol_instruct (complex multi-step synthesis)
      -- Bonus: +10 for sharegpt (diverse technical writing)
      -- Bonus: +5 for ultrachat (clear explanations)
      -- Frequency factor: log10(selection_count + 1) * 5
      ROUND(
        (avg_rating * 20) +
        CASE WHEN source = 'evol_instruct' THEN 15 ELSE 0 END +
        CASE WHEN source = 'sharegpt' THEN 10 ELSE 0 END +
        CASE WHEN source = 'ultrachat' THEN 5 ELSE 0 END +
        (LOG10(selection_count + 1) * 5),
        2
      ) AS technical_score,
      -- Explain why this model fits the weld inspection documentation use case
      CASE 
        WHEN source = 'evol_instruct' THEN 'Excellent at multi-step analysis: reviewing multiple reports, synthesizing findings, applying codes'
        WHEN source = 'sharegpt' THEN 'Strong technical writing: clear final reports, adapts to different documentation styles'
        WHEN source = 'ultrachat' THEN 'Good at explanations: connects inspection data to code requirements, detailed reasoning'
        WHEN source = 'flan_v2_niv2' THEN 'Reliable at structured tasks: follows code standards, organized report sections'
        ELSE 'Solid technical documentation performance'
      END AS use_case_fit
    FROM model_analysis
  ),
  ranked_models AS (
    SELECT 
      ROW_NUMBER() OVER (ORDER BY technical_score DESC, avg_rating DESC) AS rank,
      model,
      source,
      avg_rating,
      selection_count,
      technical_score,
      use_case_fit
    FROM scored_models
  )
  SELECT 
    rank,
    model,
    source,
    avg_rating,
    selection_count,
    technical_score,
    use_case_fit
  FROM ranked_models
  WHERE rank <= top_n
  ORDER BY rank
)

In [0]:
%sql
-- Test the technical documentation agent function
-- Use case: Find top 3 models for weld inspection report writing agent

SELECT 
  'Weld Inspection Documentation Agent' AS agent_type,
  rank,
  model,
  source,
  avg_rating,
  selection_count,
  technical_score,
  use_case_fit
FROM main.default.find_best_technical_doc_models(
  min_rating => 4.5,  -- High quality threshold
  top_n => 3          -- Top 3 recommendations
)
ORDER BY rank;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def find_best_technical_doc_models_py(min_rating, top_n):
    """
    Python implementation of find_best_technical_doc_models SQL function.
    
    Finds best model-source pairs for a technical documentation agent that:
    - Reviews weld inspection reports
    - References AWS D1.1 codes (RAG)
    - Writes final reports
    
    Args:
        min_rating (float): Minimum quality rating (e.g., 4.5)
        top_n (int): Number of recommendations to return (e.g., 3)
    
    Returns:
        DataFrame with columns: rank, model, source, avg_rating, selection_count, 
                               technical_score, use_case_fit
    """
    # Define technical sources relevant for documentation tasks
    technical_sources = ['evol_instruct', 'sharegpt', 'ultrachat', 'flan_v2_niv2']
    
    # Load and filter the assignment data
    df = spark.table("main.default.assignment_file") \
        .filter(F.col("source").isin(technical_sources))
    
    # Compute model analysis: selection count and average rating per model-source pair
    model_analysis = df.groupBy("source", F.col("`chosen-model`").alias("model")) \
        .agg(
            F.count("*").alias("selection_count"),
            F.avg("`chosen-rating`").alias("avg_rating")
        ) \
        .filter(F.col("avg_rating") >= min_rating)
    
    # Compute technical documentation score
    # Base: avg_rating * 20 (0-100 points)
    # Bonuses based on source capability fit:
    #   +15 for evol_instruct (complex multi-step synthesis)
    #   +10 for sharegpt (diverse technical writing)
    #   +5 for ultrachat (clear explanations)
    # Frequency factor: log10(selection_count + 1) * 5
    scored_models = model_analysis.withColumn(
        "technical_score",
        F.round(
            (F.col("avg_rating") * 20) +
            F.when(F.col("source") == "evol_instruct", 15)
             .when(F.col("source") == "sharegpt", 10)
             .when(F.col("source") == "ultrachat", 5)
             .otherwise(0) +
            (F.log10(F.col("selection_count") + 1) * 5),
            2
        )
    ).withColumn(
        "avg_rating",
        F.round(F.col("avg_rating"), 3)
    )
    
    # Add use case fit explanation
    scored_models = scored_models.withColumn(
        "use_case_fit",
        F.when(F.col("source") == "evol_instruct", 
               "Excellent at multi-step analysis: reviewing multiple reports, synthesizing findings, applying codes")
         .when(F.col("source") == "sharegpt",
               "Strong technical writing: clear final reports, adapts to different documentation styles")
         .when(F.col("source") == "ultrachat",
               "Good at explanations: connects inspection data to code requirements, detailed reasoning")
         .when(F.col("source") == "flan_v2_niv2",
               "Reliable at structured tasks: follows code standards, organized report sections")
         .otherwise("Solid technical documentation performance")
    )
    
    # Rank by technical_score (descending), then avg_rating (descending)
    window_spec = Window.orderBy(F.col("technical_score").desc(), F.col("avg_rating").desc())
    
    ranked_models = scored_models.withColumn(
        "rank",
        F.row_number().over(window_spec)
    ).filter(F.col("rank") <= top_n)
    
    # Select and order final columns
    result = ranked_models.select(
        "rank",
        "model",
        "source",
        "avg_rating",
        "selection_count",
        "technical_score",
        "use_case_fit"
    ).orderBy("rank")
    
    return result

print("Python function 'find_best_technical_doc_models_py' defined successfully.")
print("This is a regular Python function (not a UC function) because UC Python functions cannot access Spark tables.")

In [0]:
# Test the Python technical documentation agent function

print("=" * 80)
print("TEST: Weld Inspection Documentation Agent (Python)")
print("=" * 80)

# Call the function with default parameters
result_df = find_best_technical_doc_models_py(min_rating=4.5, top_n=3)

# Display the results
print("\nTop 3 Recommendations for Technical Documentation Agent:")
print()

# Convert to pandas for easy printing
results = result_df.toPandas()

for idx, row in results.iterrows():
    print(f"#{row['rank']} - {row['model']} ({row['source']})")
    print(f"   Rating: {row['avg_rating']}, Score: {row['technical_score']}")
    print(f"   Selection Count: {row['selection_count']}")
    print(f"   Use Case Fit: {row['use_case_fit']}")
    print()

# Also display as a table
print("\n" + "=" * 80)
print("Detailed Results Table:")
print("=" * 80)
display(result_df)

# Summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
top_rec = results.iloc[0]
print(f"\n✅ Top Recommendation: {top_rec['model']} on {top_rec['source']}")
print(f"   Technical Score: {top_rec['technical_score']}")
print(f"   Why: {top_rec['use_case_fit']}")

---
## 5. List your registered tools

Before moving on, verify all your UC functions are registered. The cell below lists functions in `main.default`.

In [0]:
%%sql
USE CATALOG main;
SHOW USER FUNCTIONS IN main.default;

## 6. Semantic Search with Embeddings *(Required)*

Semantic search lets an agent find **similar text by meaning** — not just exact keyword matches. For the UltraFeedback Expert, this means the agent can find instructions similar to a user's question, even if the wording is different.

**How it works:**
1. **Embed** each instruction using a Foundation Model embedding endpoint (`databricks-gte-large-en`).
2. **Store** the embeddings in a Delta table.
3. At query time, embed the user's question and compute **cosine similarity** against all stored embeddings.

This approach uses the Foundation Model API (available on Free Edition) instead of Vector Search endpoints (which require a quota not available on Free Edition).

> **Why not Vector Search?** Databricks Vector Search requires provisioned endpoints that exceed the Free Edition quota. The manual embedding approach teaches the same concepts (embeddings, similarity search, retrieval) and works with no extra resources.

**Docs:** [Foundation Model APIs — Embeddings](https://docs.databricks.com/en/machine-learning/model-serving/score-foundation-models.html)

### 6.1 Prepare a source table

We'll create a focused table with 100 unique instructions. This keeps embedding costs low while giving the agent plenty of material to search.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# First, check what columns are in the source table
af = spark.table("main.default.assignment_file")
print(f"assignment_file columns: {af.columns}")

# Detect the instruction column name (some versions use 'instruction', others use 'prompt')
if "instruction" in af.columns:
    text_col = "instruction"
elif "prompt" in af.columns:
    text_col = "prompt"
else:
    raise ValueError(f"Expected 'instruction' or 'prompt' column. Found: {af.columns}")

print(f"Using text column: '{text_col}'")

# Create a table of 100 unique instructions for semantic search
source_df = (
    af.select("source", af[text_col].alias("instruction"))
    .dropDuplicates(["instruction"])
    .limit(100)
    .withColumn("id", monotonically_increasing_id())
)

source_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.default.ultrafeedback_embeddings")

print(f"Created table with {spark.table('main.default.ultrafeedback_embeddings').count()} rows.")
display(spark.table("main.default.ultrafeedback_embeddings").limit(10))


### 6.2 Compute embeddings

We'll use the `databricks-gte-large-en` Foundation Model endpoint to compute embeddings. The endpoint accepts batches of text and returns vectors (1024 dimensions for GTE-Large).

We embed in batches because the API has input size limits.

In [0]:
import mlflow.deployments
import json
import time

client = mlflow.deployments.get_deploy_client("databricks")

def get_embeddings_batch(texts, endpoint="databricks-gte-large-en", max_retries=5):
    """Embed a list of texts with retry logic for rate limits."""
    for attempt in range(max_retries):
        try:
            response = client.predict(
                endpoint=endpoint,
                inputs={"input": texts}
            )
            return [item["embedding"] for item in response["data"]]
        except Exception as e:
            if "429" in str(e) or "RATE" in str(e).upper():
                wait = 2 ** attempt * 5  # 5s, 10s, 20s, 40s, 80s
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("Max retries exceeded for embedding request.")

# Load instructions from the source table
instructions_df = spark.table("main.default.ultrafeedback_embeddings").toPandas()
instructions = instructions_df["instruction"].tolist()
ids = instructions_df["id"].tolist()

print(f"Embedding {len(instructions)} instructions (with rate-limit handling)...")

# Embed in small batches with a pause between each to avoid rate limits
BATCH_SIZE = 5
SLEEP_BETWEEN = 2  # seconds between batches
all_embeddings = []

for i in range(0, len(instructions), BATCH_SIZE):
    batch = instructions[i:i + BATCH_SIZE]
    batch_embeddings = get_embeddings_batch(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"  Embedded {min(i + BATCH_SIZE, len(instructions))}/{len(instructions)}")
    if i + BATCH_SIZE < len(instructions):
        time.sleep(SLEEP_BETWEEN)

print(f"Done. {len(all_embeddings)} embeddings, {len(all_embeddings[0])} dimensions each.")


In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType, FloatType

# Build a DataFrame with id, instruction, source, and embedding
embedding_records = []
for idx, (row_id, instr, emb) in enumerate(zip(ids, instructions, all_embeddings)):
    embedding_records.append({
        "id": int(row_id),
        "instruction": instr,
        "source": instructions_df.iloc[idx]["source"],
        "embedding": emb
    })

schema = StructType([
    StructField("id", LongType(), False),
    StructField("instruction", StringType(), False),
    StructField("source", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), False),
])

emb_spark_df = spark.createDataFrame(embedding_records, schema=schema)

# Save to Delta — this is our "vector index"
emb_spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.ultrafeedback_embeddings")

print(f"Saved {len(embedding_records)} embeddings to main.default.ultrafeedback_embeddings.")
display(spark.table("main.default.ultrafeedback_embeddings").select("id", "instruction", "source").limit(3))

### 6.3 Create a similarity search function

Now we create a **SQL** UC function that the agent can call. Given a query, it uses keyword matching
(`LIKE`) against stored instructions and returns the top matches.

**NOTE** In practice this would likely be served by a Vector Search Index, but the Free Edition is limited in that capacity so this will serve as a proxy by using keyword matching to serve the same purpose

> **Why SQL instead of Python?** UC Python functions run in an isolated sandbox with no access to
> SparkSession or Databricks auth. SQL functions, by contrast, can natively query Delta tables and
> are executed by the SQL engine — no sandbox restrictions.

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.search_similar_instructions(
  query STRING COMMENT 'The search query — a word, phrase, or topic to find similar instructions in the UltraFeedback dataset. Examples: python, quantum computing, write a story.',
  top_k INT COMMENT 'Maximum number of matching instructions to return. Use 3 for a quick look or 5 for more detail.'
)
RETURNS STRING
COMMENT 'Searches UltraFeedback instructions by keyword matching and returns matching instructions with their source names. Use this to explore what kinds of prompts exist in the dataset for a given topic.'
RETURN (
  SELECT CONCAT(
    'Query: ', query, ' | Showing up to ', CAST(top_k AS STRING), ' matches:\n',
    COALESCE(
      ARRAY_JOIN(
        SLICE(
          COLLECT_LIST(CONCAT('[', source, '] ', LEFT(instruction, 200))),
          1, top_k
        ),
        '\n'
      ),
      'No matches found.'
    )
  )
  FROM main.default.ultrafeedback_embeddings
  WHERE LOWER(instruction) LIKE CONCAT('%', LOWER(query), '%')
)

### 6.4 Test the similarity search

In [0]:
%sql
-- Test: find instructions about Python
SELECT main.default.search_similar_instructions('python', 3) AS result
UNION ALL
SELECT main.default.search_similar_instructions('quantum computing', 3)
UNION ALL
SELECT main.default.search_similar_instructions('health benefits', 3)


---
## 7. Configure an external MCP server in Databricks *(Required)*

The **Model Context Protocol (MCP)** is an open standard that lets AI agents connect to external tools and data sources. Databricks supports **external MCP servers** through managed proxy endpoints — you install them from the **Databricks Marketplace** and they become available in **AI Playground** and in your agent code.

You'll install the **You.com MCP server**, which provides tools for:
- **Web search** — search the web, news, and AI-optimized results
- **Content extraction** — extract page content from URLs in markdown format

This means your agent will be able to search the web for current information about LLMs, benchmarks, and research papers — something it can't do with just the UltraFeedback dataset.

**Docs:** [External MCP Servers on Databricks](https://docs.databricks.com/aws/en/generative-ai/mcp/external-mcp) · [You.com on Databricks Marketplace](https://marketplace.databricks.com/details/32d5b7b6-0fab-4bba-9c57-5de23dd58996/Youcom_Youcom-MCP-The-1-AI-Web-Search-API) · [MCP Architecture](https://modelcontextprotocol.io/docs/learn/architecture)

### 7.1 Get a You.com API key

1. Go to [you.com/platform](https://you.com/platform).
2. Sign in or create an account.
3. Generate an API key and copy it. **Keep it safe — you'll need it in the next step.**

- Note you get $100 of complimentary credits

### 7.2 Install You.com MCP server from the Databricks Marketplace

Databricks Marketplace offers curated MCP servers that you can install with a few clicks. This creates a **Unity Catalog connection** that proxies requests to the external MCP server.

#### Steps

1. In your Databricks workspace, go to **Marketplace** → **Agents** → **MCP Servers** tab.
2. Find **You.com** (or search for it) and click **Install**.
3. In the installation dialog:
   - **Connection name**: Enter a name like `youcom_connection` (this becomes the identifier in Unity Catalog).
   - **Host / Base path**: These are pre-populated for Marketplace servers.
   - **Bearer token**: Paste your You.com API key from step 7.1.
4. Click **Install** to create the connection.

Once installed:
- A Unity Catalog connection is created with your MCP server details.
- Databricks provisions a managed proxy endpoint for secure, authenticated access.
- The proxy URL will be: `https://<workspace-hostname>/api/2.0/mcp/external/youcom_connection`

> **Tip:** To view your installed MCP servers, go to your workspace → **Agents** → **MCP Servers**.

> **Alternative (advanced):** You can also create a custom HTTP connection manually. See [External MCP Servers docs](https://docs.databricks.com/aws/en/generative-ai/mcp/external-mcp) for details.

### 7.3 Test your MCP connection in AI Playground

The fastest way to verify your MCP server is working is through **AI Playground**:

1. Go to **AI Playground** in your Databricks workspace.
2. Choose a model with the **Tools enabled** label (e.g., `databricks-meta-llama-3-3-70b-instruct`).
3. Click **Tools** → **+ Add tool** → **MCP Servers** → **External MCP servers**.
4. Select your `youcom_connection` (or whatever you named it).
5. Try a query that requires live web search:
   - *"Search the web for the latest LLM benchmarks from 2025-2026."*
   - *"What are the top open-source LLMs released in the last 6 months?"*

The model should invoke the You.com search tool and return live web results.

> **Take a screenshot** of the agent using the You.com MCP tool in AI Playground. Include it in your submission as `screenshots/mcp_you_com.png`.

> **Troubleshooting:**
> - If the MCP server doesn't appear, verify your connection was created under **Catalog** → **Connections**.
> - Check that your You.com API key is active at [you.com/platform](https://you.com/platform).
> - Ensure you have `USE CONNECTION` privilege on the connection (workspace admins have this by default).

## Short-answer Questions

Answer these questions without AI!

### Q1: When should an agent use a SQL UC function vs. a Python UC function vs. an external MCP tool?

*Write your answer below (2-3 sentences):*

SQL UC finction: When we are looking for a fast execution from the database, it can be querying or transforming structured data from the dataset.

Python UC function: If we look for extractring algorithmic tasks from a dataset like local mathematical evaluations, custom logic , and data parsing.

External MCP tool: When SQL and Python cannot do these kind of tasks for us, sometimes we need to reach outside the system to another service or workflow, we use external MPC tool. For example, it can do web searchig, finding special type of data and infomation from a third-party platform services, and when the agent requires real-time access to live web data.


### Q2: How does adding an MCP server (like You.com) extend what an agent can do beyond UC functions that point to internal data?

*Write your answer below (2-3 sentences):*

This can be like opening a window to the outside! Through SQL and Python functions, we just can search and extract information internally, however the MCP server give us an option to access external data through the Internet. There are some advantages that this method brings. First, the agent can interact with external apps, APIs, and services, second, it effectively prevents hallucination on *time-sensitive* topics that occurred after the internal database's last ETL run. 

### Q3: What functionality do these tools we've created offer that LLMs do not inherently provide?

*Write your answer below (2-3 sentences):*

LLMs are probabilistic next-token predictors that generate text based on training weights, whereas these tools provide deterministic execution. The agent can use them to do a specific task step by step and produce a valuable context. In addition, it gives us the ability to perform several tasks automatically. These tools give the agent real action-taking ability, they let it retrieve data, compute results, call services, and move work forward instead of only generating text. This also guarantees computational accuracy and real-world compliance and hallucination mitigation.

---
## 8. Bonus: Create an Agent Skill *(Optional, strongly encouraged)*

An **Agent Skill** is a markdown document (`SKILL.md`) that gives an AI assistant domain knowledge. When a skill is loaded, the assistant knows how to use specific tools, follow procedures, and avoid common mistakes — without you having to explain everything in every prompt.

Think of it as a **user manual for your agent's tools**, written so another AI can follow it.

### Why this matters

You've just built several tools (UC functions, semantic search, MCP). But an AI assistant doesn't automatically know *when* to use each one, *how* to call them, or *what to watch out for*. A skill bridges that gap.

### Create a skill

Create a file called `SKILL.md` in your `assignment_3/` folder with the following structure:

```markdown
---
name: ultrafeedback-expert
description: >
  Tools and knowledge for exploring the UltraFeedback LLM preference dataset.
  Activate when: user asks about LLM preferences, model comparisons, or
  instruction quality in the UltraFeedback dataset.
---

# UltraFeedback Expert

## When to Use This Skill

**Trigger patterns:**
- "UltraFeedback" or "preference data" or "chosen vs rejected"
- "Which model is preferred" or "model comparison"
- "Find similar instructions" or "semantic search"

## Available Tools

| Tool | Type | What it does |
|------|------|--------------|
| `main.default.lookup_source_info` | UC SQL | Returns row count and sample for a source |
| `main.default.analyze_instruction` | UC Python | Analyzes instruction complexity |
| `main.default.compare_models` | UC SQL | Compares two models by chosen vs rejected counts |
| `main.default.get_model_win_rate` | UC SQL | Returns a model's win rate in the dataset |
| `main.default.classify_instruction_topic` | UC Python | Classifies instruction topic by keywords |
| `main.default.search_similar_instructions` | UC SQL | Keyword-based search over dataset instructions |
| You.com MCP (Databricks) | External MCP | Live web search via Databricks proxy |

## Procedures

### Answering "What sources are in the dataset?"
1. Call `lookup_source_info` for each known source.
2. Summarize counts and sample instructions.

### Finding similar instructions
1. Call `search_similar_instructions` with the user's text.
2. Return the top 3-5 matches with their sources and similarity scores.

## Gotchas
- Embeddings table (`main.default.ultrafeedback_embeddings`) must exist with pre-computed vectors.
- Column names with hyphens (e.g., `chosen-model`) need backtick escaping.
```

Fill in the details based on the actual tools you created. You can use this skill in Cursor by placing it in `~/.cursor/skills/` or referencing it in a project rule.

**Docs:** [Agent Skills standard](https://github.com/xnano-ai/agentskills) · [Cursor Rules](https://cursor.com/docs/context/rules) · [Anthropic Agent Skills Guide](https://resources.anthropic.com/hubfs/The-Complete-Guide-to-Building-Skill-for-Claude.pdf?hsLang=en)

---
## Lab complete

### Required (Sections 1–7)
- [ ] **Section 3:** Verified the UltraFeedback table exists.
- [ ] **Section 4:** Created and tested the SQL functions (`all_model_combinations` and `compare_models`) and Python function (`analyze_instruction`).
- [ ] **Section 4.3:** Created and tested your own UC function.
- [ ] **Section 5:** Listed all registered UC functions.
- [ ] **Section 6:** Created embeddings for 100 instructions, registered the  UC function, and tested semantic search.
- [ ] **Section 7:** Installed the You.com MCP server from Databricks Marketplace and tested it in AI Playground (screenshot taken).

- Answer the short-answer questions prior to submitting the assignment.
### Optional but strongly encouraged (Section 8)
- [ ] **Section 8:** Created a `SKILL.md` documenting your tools.

**Submit:** Your executed notebook (`.ipynb` with all outputs) and the completed `SUBMISSION_3.md`. Include screenshots in the `screenshots/` folder.

*Next week you'll wire these tools into a working agent, register a prompt, and compare different LLMs.*